# Causal Inference: Fed Rates vs. S&P 500

Monthly observations. FEDFUNDS is a monthly average of daily effective rates, not an FOMC decision series. Record the CSV download date before reporting results.


In [ ]:
from pathlib import Path
import pandas as pd
import yfinance as yf
from casual_inference_finance.monthly import prepare_monthly_data

# Run after `uv sync`; supports kernels launched from root or notebooks/.
ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
# Exclude the current, incomplete calendar month. Fetch one prior month for returns.
end = pd.Timestamp.now(tz="America/New_York").normalize().replace(day=1)
sp500 = yf.download("^GSPC", start="1999-12-01", end=end.date().isoformat(),
                    auto_adjust=False, progress=False)
if sp500.empty:
    raise ValueError("S&P 500 download returned no prices.")
close = sp500["Close"]
if isinstance(close, pd.DataFrame):
    close = close["^GSPC"]
fed = pd.read_csv(ROOT / "data" / "FEDFUNDS.csv")
monthly = prepare_monthly_data(close, fed).loc["2000-01":]
# Keep full calendar rows until after all lags/leads have been computed.
analysis = monthly.dropna(subset=["fed_rate", "rate_change_pp", "next_month_return"])
if analysis.empty:
    raise ValueError("No complete overlapping monthly observations.")
print(f"Retrieved S&P 500: {pd.Timestamp.now(tz='UTC').isoformat()}")
print(f"Analysis: {analysis.index.min()} to {analysis.index.max()} ({len(analysis)} months)")


In [ ]:
monthly.info()
monthly.tail()


In [ ]:
# Audit missing observations before interpreting the complete-case sample.
monthly.isna().sum()


## 1. Question

How is the change in the monthly average effective federal funds rate in month t associated with the S&P 500 price return in month t+1? Unit: calendar month. A causal estimand requires additional identification assumptions.


## 2. Variables

`rate_change_pp`: monthly average rate change in percentage points. `next_month_return`: fractional next-month price return (multiply by 100 for percent). `rate_direction` describes average-rate movements, not hike/cut/hold decisions. Confounders must be available before exposure; account for publication lags and revisions.


## 3. Method

Start with descriptive monthly comparisons. A causal analysis must address policy endogeneity, anticipation, confounding, and serial dependence. Monthly averages cannot support announcement-window identification. DiD needs credible comparison units and parallel trends; synthetic control needs donors unaffected by the intervention. Method and estimand remain to be justified.


## 4. Compare the groups

_Descriptive table and plots._

In [ ]:
# Descriptive association only; these groups are not randomly assigned.
analysis.groupby("rate_direction")["next_month_return"].agg(["count", "mean", "std"])


## 5. Estimate

_Naive difference, then method-based estimate with confidence interval._

## 6. Checks

_Robustness checks._

## 7. Conclude

_Answer, limits, recommendation._